In [2]:
import pandas as pd
import numpy as np
import requests as rq
from bs4 import BeautifulSoup

In [5]:
latitude = 28.6139
longitude = 77.2090

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": latitude,
    "longitude": longitude,
    "current": "temperature_2m,relative_humidity_2m,wind_speed_10m"
}

response = rq.get(url, params=params)
data = response.json()

current = data["current"]

print("Delhi Weather")
print("Temperature:", current["temperature_2m"], "°C")
print("Humidity:", current["relative_humidity_2m"], "%")
print("Wind Speed:", current["wind_speed_10m"], "km/h")

Delhi Weather
Temperature: 28.0 °C
Humidity: 88 %
Wind Speed: 5.9 km/h


In [8]:
import os
from datetime import datetime, timezone, timedelta
from pathlib import Path

from dotenv import load_dotenv
from google.transit import gtfs_realtime_pb2

load_dotenv(Path("..") / ".env")

IST = timezone(timedelta(hours=5, minutes=30))
API_KEY = os.getenv("OTD_API_KEY")
if not API_KEY:
    raise ValueError("OTD_API_KEY is missing. Copy .env.example to .env and set your key.")

url = "https://otd.delhi.gov.in/api/realtime/VehiclePositions.pb"
params = {"key": API_KEY}
response = rq.get(url, params=params, timeout=30)

print(response.status_code)
print("Content Type:", response.headers.get("Content-Type"))
print("Response Size:", len(response.content), "bytes")

if response.status_code == 200:
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    print("Total entities:", len(feed.entity))

    for entity in feed.entity[:5]:
        print("\nEntity ID:", entity.id)

        if entity.HasField("vehicle"):
            vehicle = entity.vehicle

            print("Vehicle ID:", vehicle.vehicle.id)
            print("Trip ID:", vehicle.trip.trip_id)
            print("Route ID:", vehicle.trip.route_id)
            print("Latitude:", vehicle.position.latitude)
            print("Longitude:", vehicle.position.longitude)
            print("Speed:", vehicle.position.speed)
            print("Timestamp:", datetime.fromtimestamp(vehicle.timestamp, tz=IST))
            print("Status:", vehicle.current_status)
else:
    print("Request failed")
    print(response.text)


ValueError: OTD_API_KEY is missing. Copy .env.example to .env and set your key.